# Run-away OB stars in LOPS2

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
# Python standard
import os
import sys
import glob

# PlatoSim standard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.coordinates import SkyCoord
from astropy import units as u

# PlatoSim functions
import platosim.plot      as pt
import platosim.utilities as ut 
import platosim.starquery as sq
import platosim.smbhb     as bh
from platosim.lightcurve   import LightCurve
from platosim.utilities    import getFunctions
from platosim.matplotlibrc import setup_notebook
setup_notebook()

# Configure notebook 
from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important; }</style>"))

In [ ]:
# Change path to where you store your downloaded catalogues
path = Path(os.getenv('PLATO_WORKDIR')) / 'plato-go' / 'input'
idir = Path(os.getenv('PLATO_WORKDIR')) / 'runaway_OB_stars'

In [ ]:
ds = pd.read_feather(path / 'sources_PlatoCS_LOPS2_v1.ftr')
dn = pd.read_feather(path / 'sources_PlatoCS_LOPN1_v1.ftr')
dt = pd.concat([ds, dn])

In [ ]:
# df = pd.read_csv(fdir / 'gosc.aladin', sep='|', skip_blank_lines=True)
df = pd.read_csv(idir / 'CarreteroCastrillo_OBeRunaways_GaiaDR3.csv', sep=' ', 
                 comment='#', names=['GOCS', 'GaiaDR3', 'RA_ICRS', 'DE_ICRS'])
df

In [ ]:
names = [
    'HD 155913',
    'HD 104565',
    'ALS 18929',
    'ALS 11244',
    'HDE 229232 AB',
    'HD 155775',
    'HD 155775',
    'BD+60134',
    'CPD-342135',
    'HD 12323',
    'AB Cru',
    'HD 192639',
    'HD 94024'
]

In [ ]:
# Query sources from Gaia DR3 database
df = pd.DataFrame()
for n in names:
    try:
        dx = sq.simbadQuery(n, radius=15, maglim=21)
        dx['name'] = n
    except:
        print(n)
        pass
    else:
        df = pd.concat([df, dx.iloc[0].to_frame().T])

# Current data frame
cols = list(df)
cols.insert(0, cols.pop(cols.index('name')))
df = df.loc[:, cols]
df = df.drop(columns=['dis'])
df = df.reset_index(drop=True)

# Add galactic coordinates
gal = SkyCoord(df.ra, df.dec, frame='icrs', unit=u.deg).galactic
df['l'] = gal.l.deg
df['b'] = gal.b.deg
df

In [ ]:
fig, ax = bh.plot_aitoff(dt.iloc[::100], df)
# fig.savefig(fdir / 'aitoff_xray_binary.png', bbox_inches='tight', dpi=200)

In [ ]:
dx = df
fig, ax = pt.plotPlatoFOV('LOPS2', system='galactic', fovSize=30, 
                          ncamStars=True, ncamMap='PLATO-CS', showGalactic=True,
                          raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=150, lw=0.4, ec='w',
                          clabel=r'$\mathcal{P}$ [mag]', cmap='RdYlGn', figsize=(9,9))
ax.text(340, 670, 'LOPS2', horizontalalignment='center', verticalalignment='center', fontsize=20);
# fig.savefig(fdir / 'skymap_xray_binary_LOPS2.png', bbox_inches='tight', dpi=200)

In [ ]:
dx = df
fig, ax = pt.plotPlatoFOV('LOPN1', system='galactic', fovSize=30, 
                          ncamStars=True, ncamMap='PLATO-CS', showGalactic=True,
                          raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, lw=0.2,
                          clabel=r'$\mathcal{P}$ [mag]', cmap='PuOr', figsize=(9,9))
ax.text(340, 670, 'LOPN1', horizontalalignment='center', verticalalignment='center', fontsize=20);
# fig.savefig(fdir / 'skymap_xray_binary_LOPN1.png', bbox_inches='tight', dpi=200)